# Notebook 09 — Data Cleaning and Diverse Retraining

Addresses the cross-source generalisation gap discovered during external evaluation by cleaning salutation artefacts from generated emails and incorporating diverse training data from the Gemini BEC dataset.
Initial external testing on 4,181 Gemini-generated BEC emails revealed a severe generalisation gap, the original system detected only 2.7% 
of unseen Gemini phishing despite 99% internal accuracy. This notebook implements two corrective steps.

## What This Notebook Does

### Step 1 — Salutation Cleaning
- Removes formulaic salutations (Dear [Recipient Name], Dear Customer)from all 2,492 Qwen2.5-generated training emails
- These were artificial artefacts of the generation prompt
- Saves cleaned emails to data/generated/ai_phishing_cleaned.csv

### Step 2 — Diverse Dataset Construction
- Splits external BEC dataset 50/50 (2,090 training / 2,091 test)
- Adds 2,090 Gemini BEC emails to training as AI phishing class
- Creates diverse dataset of 12,582 emails across three classes
- Re-extracts stylometric features for all 12,582 emails

### Step 3 — Diverse XGBoost Training
- Trains XGBoost on diverse features
- Evaluates on internal test set
- Tests on held-out BEC emails (2,091 never seen during training)

## Inputs
- data/generated/ai_phishing_all.csv (from Notebook 03)
- data/processed/dataset_cleaned.csv
- data/raw/external_test/bec_dataset.csv
- data/processed/modernbert_diverse_features.csv (from Notebook 05b)
- data/processed/bec_diverse_modernbert.csv (from Notebook 05b)

## Outputs
- data/generated/ai_phishing_cleaned.csv
- data/processed/dataset_cleaned.csv
- data/processed/dataset_diverse.csv
- data/processed/stylometric_features_cleaned.csv
- data/processed/stylometric_features_diverse.csv
- data/processed/bec_test_holdout.csv
- models/xgboost_cleaned.pkl
- models/xgboost_diverse.pkl

## Key Results — BEC Detection Progression
| System | BEC Detection Rate |
|--------|-------------------|
| Original fusion | 2.7% |
| Cleaned fusion | 6.2% |
| Diverse ModernBERT alone | 99.7% |
| Diverse fusion | 99.8% |

## Conclusion
Training data diversity across AI sources is the critical factor for cross-source generalisation in AI phishing detection.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

BASE_DIR = Path("C:/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"
DATA_GENERATED = BASE_DIR / "data" / "generated"

# Load generated AI phishing emails
ai_phishing = pd.read_csv(DATA_GENERATED / "ai_phishing_all.csv")
print(f"AI phishing emails loaded: {len(ai_phishing)}")
print(f"\nSample 1:")
print(ai_phishing['text'].iloc[0][:300])
print("\nSample 2:")
print(ai_phishing['text'].iloc[1][:300])
print("\nSample 3:")
print(ai_phishing['text'].iloc[2][:300])

In [ ]:
#clean salutations
def remove_salutation(text):
    """Remove common salutation patterns from generated emails."""
    text = str(text)
    
    # Remove common salutation patterns
    patterns = [
        r"Dear\s+\[.*?\]\s*,?\s*\n+",  # Dear [Recipient's Name],
        r"Dear\s+Customer\s*,?\s*\n+",   # Dear Customer,
        r"Dear\s+User\s*,?\s*\n+",       # Dear User,
        r"Dear\s+Client\s*,?\s*\n+",     # Dear Client,
        r"Dear\s+Sir/Madam\s*,?\s*\n+",  # Dear Sir/Madam,
        r"Hello\s+\[.*?\]\s*,?\s*\n+",   # Hello [Name],
        r"Hi\s+\[.*?\]\s*,?\s*\n+",      # Hi [Name],
    ]
    
    for pattern in patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)
    
    return text.strip()

# Test on samples
print("BEFORE AND AFTER CLEANING:\n")
for i in range(3):
    original = ai_phishing['text'].iloc[i]
    cleaned = remove_salutation(original)
    print(f"Sample {i+1}:")
    print(f"  BEFORE: {original[:100]}")
    print(f"  AFTER:  {cleaned[:100]}")
    print()

In [ ]:
# Apply cleaning to all AI phishing emails
ai_phishing['text_cleaned'] = ai_phishing['text'].apply(remove_salutation)

# Check how many were actually cleaned
changed = (ai_phishing['text'] != ai_phishing['text_cleaned']).sum()
print(f"Emails modified: {changed}/{len(ai_phishing)}")
print(f"Emails unchanged: {len(ai_phishing) - changed}/{len(ai_phishing)}")

# Replace text column with cleaned version
ai_phishing['text'] = ai_phishing['text_cleaned']
ai_phishing = ai_phishing.drop(columns=['text_cleaned'])

# Verify
print(f"\nSample after cleaning:")
print(ai_phishing['text'].iloc[0][:200])

# Save cleaned file
save_path = DATA_GENERATED / "ai_phishing_cleaned.csv"
ai_phishing.to_csv(save_path, index=False)
print(f"\nSaved: {save_path}")
print(f"Emails saved: {len(ai_phishing)}")

In [ ]:
# Load original dataset components
dataset_original = pd.read_csv(DATA_PROCESSED / "dataset_final.csv")

# Separate legitimate and human phishing (keep as is)
legitimate = dataset_original[dataset_original['label'] == 0].copy()
human_phishing = dataset_original[dataset_original['label'] == 1].copy()

print(f"Legitimate emails: {len(legitimate)}")
print(f"Human phishing emails: {len(human_phishing)}")

# Load cleaned AI phishing
ai_cleaned = pd.read_csv(DATA_GENERATED / "ai_phishing_cleaned.csv")
ai_phishing_clean = ai_cleaned[['text', 'label']].copy()
ai_phishing_clean['label'] = 2

print(f"Cleaned AI phishing emails: {len(ai_phishing_clean)}")

# Combine
dataset_cleaned = pd.concat([
    legitimate,
    human_phishing,
    ai_phishing_clean
], ignore_index=True)

# Shuffle
dataset_cleaned = dataset_cleaned.sample(
    frac=1, random_state=42).reset_index(drop=True)

print(f"\nRebuilt dataset:")
print(dataset_cleaned['label'].value_counts().sort_index())

# Save
save_path = DATA_PROCESSED / "dataset_cleaned.csv"
dataset_cleaned.to_csv(save_path, index=False)
print(f"\nSaved: {save_path}")

In [ ]:
import nltk
import spacy
import textstat
import string
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nlp = spacy.load("en_core_web_sm")

# Paste all four feature functions here (same as Notebook 04)
def extract_basic_features(text):
    text = str(text); words = text.split()
    sentences = nltk.sent_tokenize(text)
    num_words = len(words) if len(words)>0 else 1
    num_sentences = len(sentences) if len(sentences)>0 else 1
    return {'email_length':len(text),'num_words':len(words),
        'num_sentences':num_sentences,
        'num_paragraphs':len([p for p in text.split('\n\n') if p.strip()]),
        'avg_word_length':np.mean([len(w) for w in words]) if words else 0,
        'max_word_length':max([len(w) for w in words]) if words else 0,
        'avg_sentence_length':num_words/num_sentences,
        'unique_words':len(set(words)),
        'type_token_ratio':len(set(words))/num_words,
        'vocab_richness':len(set(w.lower() for w in words))/num_words,
        'num_exclamations':text.count('!'),'num_questions':text.count('?'),
        'num_commas':text.count(','),'num_periods':text.count('.'),
        'exclamation_ratio':text.count('!')/num_words,
        'question_ratio':text.count('?')/num_words,
        'num_capitals':sum(1 for c in text if c.isupper()),
        'capital_ratio':sum(1 for c in text if c.isupper())/len(text) if text else 0,
        'num_special_chars':sum(1 for c in text if c in string.punctuation),
        'special_char_ratio':sum(1 for c in text if c in string.punctuation)/len(text) if text else 0}

def extract_phishing_features(text):
    text=str(text); words=text.split()
    num_words=len(words) if len(words)>0 else 1; tl=text.lower()
    urls=re.compile(r'http[s]?://(?:[a-zA-Z0-9$-_@.&+!*\\(\\),]|(?:%[0-9a-fA-F]{2}))+').findall(text)
    uw=['urgent','immediately','expire','suspend','verify','confirm','update',
        'click','login','password','account','bank','limited','offer','winner',
        'prize','free','congratulations','selected','act now']
    gw=['dear','hello','hi','greetings','good morning','good afternoon',
        'dear customer','dear user']
    tw=['suspended','terminated','blocked','restricted','unauthorized',
        'illegal','fraud','risk']
    return {'num_urls':len(urls),'has_url':int(len(urls)>0),
        'url_ratio':len(urls)/num_words,'num_http':tl.count('http://'),
        'num_https':tl.count('https://'),
        'urgency_word_count':sum(1 for w in uw if w in tl),
        'threat_word_count':sum(1 for w in tw if w in tl),
        'has_greeting':int(any(g in tl for g in gw)),
        'has_unsubscribe':int('unsubscribe' in tl),
        'has_dear':int('dear' in tl),
        'has_winner':int('winner' in tl or 'won' in tl),
        'has_free':int('free' in tl),'has_click_here':int('click here' in tl),
        'has_verify':int('verify' in tl or 'verification' in tl),
        'has_account':int('account' in tl),'has_password':int('password' in tl),
        'has_bank':int('bank' in tl),
        'has_invoice':int('invoice' in tl or 'payment' in tl),
        'num_digits':sum(c.isdigit() for c in text),
        'digit_ratio':sum(c.isdigit() for c in text)/len(text) if text else 0}

def extract_readability_features(text):
    text=str(text)
    if len(text.split())<10:
        return {k:0 for k in ['flesch_reading_ease','flesch_kincaid_grade',
            'gunning_fog','smog_index','coleman_liau_index',
            'automated_readability_index','dale_chall_readability',
            'difficult_words','linsear_write_formula','text_standard']}
    try:
        return {'flesch_reading_ease':textstat.flesch_reading_ease(text),
            'flesch_kincaid_grade':textstat.flesch_kincaid_grade(text),
            'gunning_fog':textstat.gunning_fog(text),
            'smog_index':textstat.smog_index(text),
            'coleman_liau_index':textstat.coleman_liau_index(text),
            'automated_readability_index':textstat.automated_readability_index(text),
            'dale_chall_readability':textstat.dale_chall_readability_score(text),
            'difficult_words':textstat.difficult_words(text),
            'linsear_write_formula':textstat.linsear_write_formula(text),
            'text_standard':float(str(textstat.text_standard(text,float_output=True)))}
    except:
        return {k:0 for k in ['flesch_reading_ease','flesch_kincaid_grade',
            'gunning_fog','smog_index','coleman_liau_index',
            'automated_readability_index','dale_chall_readability',
            'difficult_words','linsear_write_formula','text_standard']}

def extract_syntactic_features(text):
    text=str(text)
    if len(text)>5000: text=text[:5000]
    doc=nlp(text); tt=len(doc) if len(doc)>0 else 1
    pc={}
    for token in doc: pc[token.pos_]=pc.get(token.pos_,0)+1
    return {'noun_ratio':pc.get('NOUN',0)/tt,'verb_ratio':pc.get('VERB',0)/tt,
        'adj_ratio':pc.get('ADJ',0)/tt,'adv_ratio':pc.get('ADV',0)/tt,
        'pronoun_ratio':pc.get('PRON',0)/tt,'propn_ratio':pc.get('PROPN',0)/tt,
        'det_ratio':pc.get('DET',0)/tt,'punct_ratio':pc.get('PUNCT',0)/tt,
        'num_ratio':pc.get('NUM',0)/tt,'num_entities':len(doc.ents),
        'entity_ratio':len(doc.ents)/tt,
        'stopword_ratio':sum(1 for t in doc if t.is_stop)/tt,
        'unique_punct':len(set(t.text for t in doc if t.is_punct))}

def extract_all_features(text):
    f={}; f.update(extract_basic_features(text))
    f.update(extract_phishing_features(text))
    f.update(extract_readability_features(text))
    f.update(extract_syntactic_features(text))
    return f

print("Extracting stylometric features for cleaned dataset")

all_features = []
for idx, row in tqdm(dataset_cleaned.iterrows(), total=len(dataset_cleaned)):
    try:
        features = extract_all_features(row['text'])
        features['label'] = row['label']
        all_features.append(features)
    except Exception as e:
        continue

features_cleaned_df = pd.DataFrame(all_features)
save_path = DATA_PROCESSED / "stylometric_features_cleaned.csv"
features_cleaned_df.to_csv(save_path, index=False)

print(f"\nDone!")
print(f"Shape: {features_cleaned_df.shape}")
print(f"Saved: {save_path}")

In [ ]:
#retrain XGBoost with cleaned features
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
import xgboost as xgb
import pickle
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path("C:/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"

# Load cleaned features
stylo_cleaned = pd.read_csv(DATA_PROCESSED / "stylometric_features_cleaned.csv")
modernbert_cleaned = pd.read_csv(DATA_PROCESSED / "modernbert_cleaned_features.csv")

print(f"Stylometric (cleaned): {stylo_cleaned.shape}")
print(f"ModernBERT (cleaned): {modernbert_cleaned.shape}")

# Build feature matrix
stylo_features = stylo_cleaned.drop(columns=['label'])
modernbert_scores = modernbert_cleaned[['conf_legitimate',
                                        'conf_human_phishing',
                                        'conf_ai_phishing']]
y = stylo_cleaned['label']

X_fusion = pd.concat([stylo_features, modernbert_scores], axis=1)
print(f"\nFusion matrix: {X_fusion.shape}")

# Same split
X_train, X_temp, y_train, y_temp = train_test_split(
    X_fusion, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}")

# Class weights
cc = y_train.value_counts().sort_index()
total = len(y_train)
cw = {cls: total/(len(cc)*count) for cls, count in cc.items()}
sw = y_train.map(cw)

# Train
print("\nTraining XGBoost on cleaned features")
model_cleaned = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric='mlogloss', verbosity=0)

model_cleaned.fit(X_train, y_train, sample_weight=sw)

# Test set evaluation
test_preds = model_cleaned.predict(X_test)
test_proba = model_cleaned.predict_proba(X_test)

print("\nTest Set Results (Cleaned Dataset):")
print(classification_report(y_test, test_preds,
      target_names=['Legitimate', 'Human Phishing', 'AI Phishing']))
print(f"Macro F1: {f1_score(y_test, test_preds, average='macro'):.4f}")

# Save
with open(MODELS_DIR / "xgboost_cleaned.pkl", 'wb') as f:
    pickle.dump(model_cleaned, f)
print("Model saved.")

In [ ]:
#external BEC test on cleaned model
# Load BEC features
bec_stylo = pd.read_csv(DATA_PROCESSED / "bec_stylometric.csv")
bec_modernbert_cleaned = pd.read_csv(DATA_PROCESSED / "bec_modernbert_cleaned.csv")

min_len = min(len(bec_stylo), len(bec_modernbert_cleaned))
bec_stylo = bec_stylo.iloc[:min_len].reset_index(drop=True)
bec_modernbert_cleaned = bec_modernbert_cleaned.iloc[:min_len].reset_index(drop=True)

# Align columns
stylo_cols = stylo_features.columns
bec_stylo_aligned = bec_stylo[stylo_cols]

# Build BEC fusion matrix
bec_fusion = pd.concat([
    bec_stylo_aligned.reset_index(drop=True),
    bec_modernbert_cleaned[['conf_legitimate',
                             'conf_human_phishing',
                             'conf_ai_phishing']].reset_index(drop=True)
], axis=1)

# Predict
bec_preds = model_cleaned.predict(bec_fusion)
flagged = (bec_preds != 0).sum()

print("="*60)
print("EXTERNAL BEC TEST — Cleaned Model vs Original")
print("="*60)
print(f"\nCleaned Fusion system:")
print(f"  Flagged: {flagged}/{min_len} ({flagged/min_len*100:.1f}%)")
print(f"\nComparison:")
print(f"  Original ModernBERT alone:    6.1%")
print(f"  Original Fusion:              2.7%")
print(f"  Cleaned ModernBERT alone:     10.8%")
print(f"  Cleaned Fusion (this):        {flagged/min_len*100:.1f}%")

In [ ]:
#Split BEC and Build Combined Dataset

# Load BEC dataset
bec_raw = pd.read_csv(BASE_DIR / "data" / "raw" / "external_test" / "bec_dataset.csv")
bec_raw['text'] = bec_raw['subject'].astype(str) + " " + bec_raw['body'].astype(str)
bec_raw = bec_raw.dropna(subset=['label']).reset_index(drop=True)
print(f"BEC emails loaded: {len(bec_raw)}")

# Split BEC 50/50
from sklearn.model_selection import train_test_split

bec_train, bec_test = train_test_split(
    bec_raw, test_size=0.50, random_state=42)

print(f"BEC training half: {len(bec_train)}")
print(f"BEC test half: {len(bec_test)}")

# Prepare BEC training portion as AI phishing (label 2)
bec_train_clean = bec_train[['text']].copy()
bec_train_clean['label'] = 2

# Load cleaned dataset
dataset_cleaned = pd.read_csv(DATA_PROCESSED / "dataset_cleaned.csv")
print(f"\nExisting cleaned dataset: {len(dataset_cleaned)}")

# Combine
dataset_diverse = pd.concat([
    dataset_cleaned,
    bec_train_clean
], ignore_index=True)

# Shuffle
dataset_diverse = dataset_diverse.sample(
    frac=1, random_state=42).reset_index(drop=True)

print(f"\nDiverse dataset:")
print(dataset_diverse['label'].value_counts().sort_index())

# Save
dataset_diverse.to_csv(DATA_PROCESSED / "dataset_diverse.csv", index=False)
bec_test[['text', 'label']].to_csv(
    DATA_PROCESSED / "bec_test_holdout.csv", index=False)

print(f"\nSaved dataset_diverse.csv")
print(f"Saved bec_test_holdout.csv")

In [ ]:
import nltk
import spacy
import textstat
import string
import re
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nlp = spacy.load("en_core_web_sm")

def extract_basic_features(text):
    text = str(text); words = text.split()
    sentences = nltk.sent_tokenize(text)
    num_words = len(words) if len(words)>0 else 1
    num_sentences = len(sentences) if len(sentences)>0 else 1
    return {'email_length':len(text),'num_words':len(words),
        'num_sentences':num_sentences,
        'num_paragraphs':len([p for p in text.split('\n\n') if p.strip()]),
        'avg_word_length':np.mean([len(w) for w in words]) if words else 0,
        'max_word_length':max([len(w) for w in words]) if words else 0,
        'avg_sentence_length':num_words/num_sentences,
        'unique_words':len(set(words)),
        'type_token_ratio':len(set(words))/num_words,
        'vocab_richness':len(set(w.lower() for w in words))/num_words,
        'num_exclamations':text.count('!'),'num_questions':text.count('?'),
        'num_commas':text.count(','),'num_periods':text.count('.'),
        'exclamation_ratio':text.count('!')/num_words,
        'question_ratio':text.count('?')/num_words,
        'num_capitals':sum(1 for c in text if c.isupper()),
        'capital_ratio':sum(1 for c in text if c.isupper())/len(text) if text else 0,
        'num_special_chars':sum(1 for c in text if c in string.punctuation),
        'special_char_ratio':sum(1 for c in text if c in string.punctuation)/len(text) if text else 0}

def extract_phishing_features(text):
    text=str(text); words=text.split()
    num_words=len(words) if len(words)>0 else 1; tl=text.lower()
    urls=re.compile(r'http[s]?://(?:[a-zA-Z0-9$-_@.&+!*\\(\\),]|(?:%[0-9a-fA-F]{2}))+').findall(text)
    uw=['urgent','immediately','expire','suspend','verify','confirm','update',
        'click','login','password','account','bank','limited','offer','winner',
        'prize','free','congratulations','selected','act now']
    gw=['dear','hello','hi','greetings','good morning','good afternoon',
        'dear customer','dear user']
    tw=['suspended','terminated','blocked','restricted','unauthorized',
        'illegal','fraud','risk']
    return {'num_urls':len(urls),'has_url':int(len(urls)>0),
        'url_ratio':len(urls)/num_words,'num_http':tl.count('http://'),
        'num_https':tl.count('https://'),
        'urgency_word_count':sum(1 for w in uw if w in tl),
        'threat_word_count':sum(1 for w in tw if w in tl),
        'has_greeting':int(any(g in tl for g in gw)),
        'has_unsubscribe':int('unsubscribe' in tl),
        'has_dear':int('dear' in tl),
        'has_winner':int('winner' in tl or 'won' in tl),
        'has_free':int('free' in tl),'has_click_here':int('click here' in tl),
        'has_verify':int('verify' in tl or 'verification' in tl),
        'has_account':int('account' in tl),'has_password':int('password' in tl),
        'has_bank':int('bank' in tl),
        'has_invoice':int('invoice' in tl or 'payment' in tl),
        'num_digits':sum(c.isdigit() for c in text),
        'digit_ratio':sum(c.isdigit() for c in text)/len(text) if text else 0}

def extract_readability_features(text):
    text=str(text)
    if len(text.split())<10:
        return {k:0 for k in ['flesch_reading_ease','flesch_kincaid_grade',
            'gunning_fog','smog_index','coleman_liau_index',
            'automated_readability_index','dale_chall_readability',
            'difficult_words','linsear_write_formula','text_standard']}
    try:
        return {'flesch_reading_ease':textstat.flesch_reading_ease(text),
            'flesch_kincaid_grade':textstat.flesch_kincaid_grade(text),
            'gunning_fog':textstat.gunning_fog(text),
            'smog_index':textstat.smog_index(text),
            'coleman_liau_index':textstat.coleman_liau_index(text),
            'automated_readability_index':textstat.automated_readability_index(text),
            'dale_chall_readability':textstat.dale_chall_readability_score(text),
            'difficult_words':textstat.difficult_words(text),
            'linsear_write_formula':textstat.linsear_write_formula(text),
            'text_standard':float(str(textstat.text_standard(text,float_output=True)))}
    except:
        return {k:0 for k in ['flesch_reading_ease','flesch_kincaid_grade',
            'gunning_fog','smog_index','coleman_liau_index',
            'automated_readability_index','dale_chall_readability',
            'difficult_words','linsear_write_formula','text_standard']}

def extract_syntactic_features(text):
    text=str(text)
    if len(text)>5000: text=text[:5000]
    doc=nlp(text); tt=len(doc) if len(doc)>0 else 1
    pc={}
    for token in doc: pc[token.pos_]=pc.get(token.pos_,0)+1
    return {'noun_ratio':pc.get('NOUN',0)/tt,'verb_ratio':pc.get('VERB',0)/tt,
        'adj_ratio':pc.get('ADJ',0)/tt,'adv_ratio':pc.get('ADV',0)/tt,
        'pronoun_ratio':pc.get('PRON',0)/tt,'propn_ratio':pc.get('PROPN',0)/tt,
        'det_ratio':pc.get('DET',0)/tt,'punct_ratio':pc.get('PUNCT',0)/tt,
        'num_ratio':pc.get('NUM',0)/tt,'num_entities':len(doc.ents),
        'entity_ratio':len(doc.ents)/tt,
        'stopword_ratio':sum(1 for t in doc if t.is_stop)/tt,
        'unique_punct':len(set(t.text for t in doc if t.is_punct))}

def extract_all_features(text):
    f={}
    f.update(extract_basic_features(text))
    f.update(extract_phishing_features(text))
    f.update(extract_readability_features(text))
    f.update(extract_syntactic_features(text))
    return f

print("All feature functions loaded — ready to extract")

In [ ]:
print("Extracting stylometric features for diverse dataset")

all_features = []
for idx, row in tqdm(dataset_diverse.iterrows(), total=len(dataset_diverse)):
    try:
        features = extract_all_features(row['text'])
        features['label'] = row['label']
        all_features.append(features)
    except Exception as e:
        continue

features_diverse_df = pd.DataFrame(all_features)
save_path = DATA_PROCESSED / "stylometric_features_diverse.csv"
features_diverse_df.to_csv(save_path, index=False)

print(f"\nDone!")
print(f"Shape: {features_diverse_df.shape}")
print(f"Saved: {save_path}")

In [ ]:
#Retrain XGBoost with Diverse Features
# Load diverse features
stylo_diverse = pd.read_csv(DATA_PROCESSED / "stylometric_features_diverse.csv")
modernbert_diverse = pd.read_csv(DATA_PROCESSED / "modernbert_diverse_features.csv")

print(f"Stylometric (diverse): {stylo_diverse.shape}")
print(f"ModernBERT (diverse): {modernbert_diverse.shape}")

# Build feature matrix
stylo_features_div = stylo_diverse.drop(columns=['label'])
modernbert_scores_div = modernbert_diverse[['conf_legitimate',
                                            'conf_human_phishing',
                                            'conf_ai_phishing']]
y_div = stylo_diverse['label']

X_fusion_div = pd.concat([stylo_features_div, modernbert_scores_div], axis=1)
print(f"Fusion matrix: {X_fusion_div.shape}")

# Same split
X_train, X_temp, y_train, y_temp = train_test_split(
    X_fusion_div, y_div, test_size=0.30, random_state=42, stratify=y_div)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}")

# Class weights
cc = y_train.value_counts().sort_index()
total = len(y_train)
cw = {cls: total/(len(cc)*count) for cls, count in cc.items()}
sw = y_train.map(cw)

# Train
print("\nTraining XGBoost on diverse features")
model_diverse = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric='mlogloss', verbosity=0)

model_diverse.fit(X_train, y_train, sample_weight=sw)

# Test set evaluation
test_preds = model_diverse.predict(X_test)
print("\nTest Set Results (Diverse Dataset):")
print(classification_report(y_test, test_preds,
      target_names=['Legitimate', 'Human Phishing', 'AI Phishing']))
print(f"Macro F1: {f1_score(y_test, test_preds, average='macro'):.4f}")

# Save
with open(MODELS_DIR / "xgboost_diverse.pkl", 'wb') as f:
    pickle.dump(model_diverse, f)
print("Model saved.")

In [ ]:
# BEC Holdout Test on Diverse Fusion
# Load BEC holdout stylometric features
# Need to extract them first
bec_holdout = pd.read_csv(DATA_PROCESSED / "bec_test_holdout.csv")
bec_holdout['text'] = bec_holdout['text'].astype(str)
print(f"BEC holdout: {len(bec_holdout)} emails")

# Extract stylometric features for holdout
print("Extracting stylometric features for BEC holdout")
bec_holdout_features = []
for idx, row in tqdm(bec_holdout.iterrows(), total=len(bec_holdout)):
    try:
        features = extract_all_features(row['text'])
        bec_holdout_features.append(features)
    except:
        continue

bec_holdout_stylo = pd.DataFrame(bec_holdout_features)
print(f"Features extracted: {bec_holdout_stylo.shape}")

# Load ModernBERT scores for holdout
bec_diverse_mb = pd.read_csv(DATA_PROCESSED / "bec_diverse_modernbert.csv")

min_len = min(len(bec_holdout_stylo), len(bec_diverse_mb))
bec_holdout_stylo = bec_holdout_stylo.iloc[:min_len]
bec_diverse_mb = bec_diverse_mb.iloc[:min_len]

# Build fusion matrix
stylo_cols = stylo_features_div.columns
bec_fusion_holdout = pd.concat([
    bec_holdout_stylo[stylo_cols].reset_index(drop=True),
    bec_diverse_mb[['conf_legitimate',
                    'conf_human_phishing',
                    'conf_ai_phishing']].reset_index(drop=True)
], axis=1)

# Predict
fusion_preds = model_diverse.predict(bec_fusion_holdout)
flagged = (fusion_preds != 0).sum()

print("\n" + "="*60)
print("FINAL COMPARISON — All Systems on BEC Phishing")
print("="*60)
print(f"\n  Original fusion (unseen BEC):      2.7%")
print(f"  Cleaned fusion (unseen BEC):       6.2%")
print(f"  Diverse ModernBERT alone:          99.7%")
print(f"  Diverse fusion (holdout BEC):      {flagged/min_len*100:.1f}%")